<a href="https://colab.research.google.com/github/samsuseelan/ga-unstable-approach-warning/blob/main/stable_approach_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Stable approach check
# Based on Flight Safety Foundation stabilized approach criteria

def check_approach(airspeed, target_speed, sink_rate, altitude_agl):
    warnings = []

    if airspeed > target_speed + 10:
        warnings.append("Too fast")
    if airspeed < target_speed - 5:
        warnings.append("Too slow")
    if sink_rate > 1000:
        warnings.append("Sink rate too high")

    if warnings and altitude_agl < 500:
        return "UNSTABLE: go around. " + ", ".join(warnings)
    if warnings:
        return "Caution: " + ", ".join(warnings)
    return "Stable"


print(check_approach(airspeed=68, target_speed=65, sink_rate=1200, altitude_agl=400))

In [ ]:
import pandas as pd

# Practice data: 2 students, 3 lessons each
approaches = pd.DataFrame({
    "student": ["Alex", "Alex", "Alex", "Priya", "Priya", "Priya"],
    "lesson": [1, 2, 3, 1, 2, 3],
    "airspeed": [80, 74, 68, 66, 79, 82],
    "target_speed": [65, 65, 65, 65, 65, 65],
    "sink_rate": [1200, 900, 700, 650, 950, 1100],
    "altitude_agl": [400, 400, 400, 400, 400, 400],
})

# Check each approach, one row at a time
results = []
for index, row in approaches.iterrows():
    result = check_approach(row["airspeed"], row["target_speed"], row["sink_rate"], row["altitude_agl"])
    results.append(result)

approaches["result"] = results
approaches

In [ ]:
# Mark each approach as unstable (True) or not (False)
approaches["unstable"] = approaches["result"].str.startswith("UNSTABLE")

# Compare each student's first lesson with their last lesson
for student in approaches["student"].unique():
    rows = approaches[approaches["student"] == student]
    first = rows.iloc[0]["unstable"]
    last = rows.iloc[-1]["unstable"]

    if last and not first:
        print(student + ": getting worse. Review before solo.")
    elif first and not last:
        print(student + ": improving.")
    else:
        print(student + ": no change.")

In [ ]:
# Download one hour of OpenSky data: Monday 27 June 2022, 15:00 UTC
!wget -q https://s3.opensky-network.org/data-samples/states/2022-06-27/15/states_2022-06-27-15.csv.tar

# Unpack the file
!tar -xf states_2022-06-27-15.csv.tar

# List the files, with sizes
!ls -lh

In [ ]:
import pandas as pd

# Columns we need
cols = ["time", "icao24", "lat", "lon", "velocity", "heading",
        "vertrate", "callsign", "onground", "baroaltitude", "geoaltitude"]

# Box around Daytona Beach airport (KDAB), about 5 miles each side
lat_min, lat_max = 29.10, 29.26
lon_min, lon_max = -81.15, -80.97

# Read the big file in pieces of 1 million rows, keep only rows inside the box
parts = []
for chunk in pd.read_csv("states_2022-06-27-15.csv.gz", usecols=cols, chunksize=1_000_000):
    near = chunk[chunk["lat"].between(lat_min, lat_max) & chunk["lon"].between(lon_min, lon_max)]
    parts.append(near)

kdab = pd.concat(parts)

print("Rows near KDAB:", len(kdab))
print("Aircraft near KDAB:", kdab["icao24"].nunique())
kdab.head()

In [ ]:
kdab = kdab.copy()

# Convert units: meters to feet, m/s to knots, m/s to feet per minute
kdab["alt_ft"] = kdab["baroaltitude"] * 3.281
kdab["speed_kt"] = kdab["velocity"] * 1.944
kdab["vs_fpm"] = kdab["vertrate"] * 196.85

# Keep airborne aircraft below 1,500 ft and slower than 140 knots
low = kdab[(kdab["alt_ft"] < 1500) & (kdab["speed_kt"] < 140) & (kdab["onground"] == False)]

# One summary line per aircraft
summary = low.groupby("callsign").agg(
    points=("time", "count"),
    lowest_ft=("alt_ft", "min"),
    avg_speed_kt=("speed_kt", "mean"),
).round(0)

print("Aircraft in the pattern:", len(summary))
summary.sort_values("points", ascending=False)

In [ ]:
import matplotlib.pyplot as plt

# All reports for one aircraft, in time order
plane = kdab[kdab["callsign"].str.strip() == "N779G"].sort_values("time")

# Minutes since first report
minutes = (plane["time"] - plane["time"].min()) / 60

# Two charts, one above the other
fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax[0].plot(minutes, plane["alt_ft"])
ax[0].set_ylabel("Altitude (ft)")
ax[1].plot(minutes, plane["speed_kt"])
ax[1].set_ylabel("Ground speed (kt)")
ax[1].set_xlabel("Minutes")
plt.suptitle("N779G at Daytona Beach, 27 June 2022, 15:00 to 16:00 UTC")
plt.show()

In [ ]:
plane = plane.copy()

# A point belongs to an approach if the aircraft descends faster than 100 fpm below 700 ft
plane["descending"] = (plane["vs_fpm"] < -100) & (plane["alt_ft"] < 700)

# Give each run of descending points its own number
starts = plane["descending"] & ~plane["descending"].shift(fill_value=False)
plane["approach_id"] = starts.cumsum()

# Keep only descending points
descents = plane[plane["descending"]]

# One line per approach
approach_table = descents.groupby("approach_id").agg(
    points=("time", "count"),
    start_ft=("alt_ft", "max"),
    end_ft=("alt_ft", "min"),
    avg_gs_kt=("speed_kt", "mean"),
    max_sink_fpm=("vs_fpm", "min"),
).round(0)

# Drop very short runs, fewer than 3 points
approach_table = approach_table[approach_table["points"] >= 3]

print("Approaches found:", len(approach_table))
approach_table

In [ ]:
# Show every point in approach 3
plane[plane["approach_id"] == 3][["time", "alt_ft", "speed_kt", "vs_fpm", "onground"]]

In [ ]:
# Remove frozen rows: same altitude, speed, and sink rate as the row before
plane = plane.sort_values("time")
frozen = (
    (plane["alt_ft"] == plane["alt_ft"].shift()) &
    (plane["speed_kt"] == plane["speed_kt"].shift()) &
    (plane["vs_fpm"] == plane["vs_fpm"].shift())
)
print("Frozen rows removed:", frozen.sum())
clean = plane[~frozen].copy()

# Cut approaches again on the clean data
clean["descending"] = (clean["vs_fpm"] < -100) & (clean["alt_ft"] < 700)
starts = clean["descending"] & ~clean["descending"].shift(fill_value=False)
clean["approach_id"] = starts.cumsum()
descents = clean[clean["descending"]]

approach_table = descents.groupby("approach_id").agg(
    points=("time", "count"),
    start_ft=("alt_ft", "max"),
    end_ft=("alt_ft", "min"),
    avg_gs_kt=("speed_kt", "mean"),
    max_sink_fpm=("vs_fpm", "min"),
).round(0)
approach_table = approach_table[approach_table["points"] >= 3]

print("Approaches found:", len(approach_table))
approach_table

In [ ]:
def find_approaches(df):
    """Clean one aircraft's track and cut the track into approaches."""
    df = df.sort_values("time")

    # Remove frozen rows
    frozen = (
        (df["alt_ft"] == df["alt_ft"].shift()) &
        (df["speed_kt"] == df["speed_kt"].shift()) &
        (df["vs_fpm"] == df["vs_fpm"].shift())
    )
    df = df[~frozen].copy()

    # Cut approaches
    df["descending"] = (df["vs_fpm"] < -100) & (df["alt_ft"] < 700)
    starts = df["descending"] & ~df["descending"].shift(fill_value=False)
    df["approach_id"] = starts.cumsum()
    descents = df[df["descending"]]

    table = descents.groupby("approach_id").agg(
        points=("time", "count"),
        start_ft=("alt_ft", "max"),
        end_ft=("alt_ft", "min"),
        avg_gs_kt=("speed_kt", "mean"),
        max_sink_fpm=("vs_fpm", "min"),
    ).round(0)
    return table[table["points"] >= 3]


# Run the function on every aircraft in the pattern
all_tables = []
for callsign in summary.index:
    rows = kdab[kdab["callsign"] == callsign]
    table = find_approaches(rows)
    table["callsign"] = callsign
    all_tables.append(table)

all_approaches = pd.concat(all_tables).reset_index()

print("Total approaches:", len(all_approaches))
print("Aircraft with approaches:", all_approaches["callsign"].nunique())
all_approaches

In [ ]:
# Flag approaches with a peak sink rate over 1,000 fpm
all_approaches["sink_flag"] = all_approaches["max_sink_fpm"] < -1000

# One line per aircraft
per_aircraft = all_approaches.groupby("callsign").agg(
    approaches=("approach_id", "count"),
    flagged=("sink_flag", "sum"),
    avg_gs_kt=("avg_gs_kt", "mean"),
    worst_sink_fpm=("max_sink_fpm", "min"),
).round(0)

per_aircraft.sort_values("flagged", ascending=False)

In [ ]:
# Download the aircraft database for June 2022
!wget -q https://s3.opensky-network.org/data-samples/metadata/aircraft-database-complete-2022-06.csv

# Show the first 3 lines, to see the column names
!head -3 aircraft-database-complete-2022-06.csv

In [ ]:
# Load the database without column names, every value as text
db = pd.read_csv("aircraft-database-complete-2022-06.csv",
                                  header=None, skiprows=1, dtype=str, on_bad_lines="skip", encoding="latin-1")

print("Aircraft in database:", len(db))

# Find N779G and show every column, one per line
db[db[1] == "N779G"].T

In [ ]:
# Keep the useful columns and give them names
aircraft = db[[0, 3, 4]].copy()
aircraft.columns = ["icao24", "manufacturer", "model"]
aircraft = aircraft.drop_duplicates("icao24")

# Link each callsign to its icao24 code, from the flight data
ids = kdab[["callsign", "icao24"]].drop_duplicates("callsign")

# Join: approaches + icao24 + aircraft type
typed = all_approaches.merge(ids, on="callsign", how="left")
typed = typed.merge(aircraft, on="icao24", how="left")

# One line per aircraft, now with type
per_type = typed.groupby(["callsign", "manufacturer", "model"], dropna=False).agg(
    approaches=("approach_id", "count"),
    flagged=("sink_flag", "sum"),
    avg_gs_kt=("avg_gs_kt", "mean"),
    worst_sink_fpm=("max_sink_fpm", "min"),
).round(0)

per_type.sort_values("flagged", ascending=False)

In [ ]:
def find_approaches(df):
    """Clean one aircraft's track, cut into approaches, count high sink reports."""
    df = df.sort_values("time")
    frozen = (
        (df["alt_ft"] == df["alt_ft"].shift()) &
        (df["speed_kt"] == df["speed_kt"].shift()) &
        (df["vs_fpm"] == df["vs_fpm"].shift())
    )
    df = df[~frozen].copy()

    df["descending"] = (df["vs_fpm"] < -100) & (df["alt_ft"] < 700)
    starts = df["descending"] & ~df["descending"].shift(fill_value=False)
    df["approach_id"] = starts.cumsum()
    descents = df[df["descending"]]

    table = descents.groupby("approach_id").agg(
        points=("time", "count"),
        max_sink_fpm=("vs_fpm", "min"),
        high_sink_points=("vs_fpm", lambda s: (s < -1000).sum()),
    )
    return table[table["points"] >= 3]


all_tables = []
for callsign in summary.index:
    table = find_approaches(kdab[kdab["callsign"] == callsign])
    table["callsign"] = callsign
    all_tables.append(table)

all_approaches = pd.concat(all_tables).reset_index()
all_approaches["peak_flag"] = all_approaches["max_sink_fpm"] < -1000
all_approaches["sustained_flag"] = all_approaches["high_sink_points"] >= 2

typed = all_approaches.merge(ids, on="callsign", how="left").merge(aircraft, on="icao24", how="left")

per_type = typed.groupby(["callsign", "model"], dropna=False).agg(
    approaches=("approach_id", "count"),
    peak_flags=("peak_flag", "sum"),
    sustained_flags=("sustained_flag", "sum"),
    worst_sink_fpm=("max_sink_fpm", "min"),
)
per_type.sort_values("sustained_flags", ascending=False)

In [ ]:
import os

hours = ["13", "14", "15", "16", "17", "18"]
all_hours = []

for h in hours:
    name = f"states_2022-06-27-{h}.csv"
    url = f"https://s3.opensky-network.org/data-samples/states/2022-06-27/{h}/{name}.tar"

    # Download and unpack one hour
    os.system(f"wget -q {url} && tar -xf {name}.tar")

    # Keep only rows near KDAB
    parts = []
    for chunk in pd.read_csv(name + ".gz", usecols=cols, chunksize=1_000_000):
        near = chunk[chunk["lat"].between(lat_min, lat_max) & chunk["lon"].between(lon_min, lon_max)]
        parts.append(near)
    all_hours.append(pd.concat(parts))

    # Delete the big files to free disk space
    os.system(f"rm -f {name}.tar {name}.gz")
    print("Hour", h, "done")

day = pd.concat(all_hours)
print("Rows near KDAB:", len(day))
print("Aircraft near KDAB:", day["icao24"].nunique())

In [ ]:
day = day.copy()

# Convert units
day["alt_ft"] = day["baroaltitude"] * 3.281
day["speed_kt"] = day["velocity"] * 1.944
day["vs_fpm"] = day["vertrate"] * 196.85

# Aircraft seen low and slow at least once
low_day = day[(day["alt_ft"] < 1500) & (day["speed_kt"] < 140) & (day["onground"] == False)]
pattern_ids = low_day["icao24"].unique()

# Run the approach finder on each aircraft
all_tables = []
for plane_id in pattern_ids:
    table = find_approaches(day[day["icao24"] == plane_id])
    table["icao24"] = plane_id
    all_tables.append(table)

day_approaches = pd.concat(all_tables).reset_index()
day_approaches["sustained_flag"] = day_approaches["high_sink_points"] >= 2

# Add aircraft type
day_approaches = day_approaches.merge(aircraft, on="icao24", how="left")

print("Aircraft in the pattern:", len(pattern_ids))
print("Total approaches:", len(day_approaches))
print("Sustained flags:", day_approaches["sustained_flag"].sum())

# One line per aircraft, worst first
per_aircraft = day_approaches.groupby(["icao24", "model"], dropna=False).agg(
    approaches=("approach_id", "count"),
    sustained_flags=("sustained_flag", "sum"),
    worst_sink_fpm=("max_sink_fpm", "min"),
).round(0)

per_aircraft.sort_values("sustained_flags", ascending=False).head(15)

In [ ]:
# ac26ef reports, frozen rows removed
plane = day[day["icao24"] == "ac26ef"].sort_values("time")
frozen = (
    (plane["alt_ft"] == plane["alt_ft"].shift()) &
    (plane["speed_kt"] == plane["speed_kt"].shift()) &
    (plane["vs_fpm"] == plane["vs_fpm"].shift())
)
plane = plane[~frozen].copy()

# High sink reports below 700 ft, with clock time
high = plane[(plane["vs_fpm"] < -1000) & (plane["alt_ft"] < 700)].copy()
high["utc"] = pd.to_datetime(high["time"], unit="s")

high[["utc", "callsign", "alt_ft", "speed_kt", "vs_fpm"]]

In [ ]:
def find_approaches_v2(df):
    """Clean one track, cut approaches, count back-to-back high sink reports above 100 ft."""
    df = df.sort_values("time")
    frozen = (
        (df["alt_ft"] == df["alt_ft"].shift()) &
        (df["speed_kt"] == df["speed_kt"].shift()) &
        (df["vs_fpm"] == df["vs_fpm"].shift())
    )
    df = df[~frozen].copy()

    df["descending"] = (df["vs_fpm"] < -100) & (df["alt_ft"] < 700)
    starts = df["descending"] & ~df["descending"].shift(fill_value=False)
    df["approach_id"] = starts.cumsum()
    descents = df[df["descending"]].copy()

    # High sink, ignoring the last 100 ft before touchdown
    descents["high"] = (descents["vs_fpm"] < -1000) & (descents["alt_ft"] > 100)

    # True when this report and the one 10 seconds before are both high
    prev_high = descents.groupby("approach_id")["high"].shift(fill_value=False)
    gap_ok = descents["time"].diff() <= 15
    descents["back_to_back"] = descents["high"] & prev_high & gap_ok

    table = descents.groupby("approach_id").agg(
        points=("time", "count"),
        start_utc=("time", "min"),
        max_sink_fpm=("vs_fpm", "min"),
        back_to_back=("back_to_back", "sum"),
    )
    return table[table["points"] >= 3]


v2_tables = []
for plane_id in pattern_ids:
    table = find_approaches_v2(day[day["icao24"] == plane_id])
    table["icao24"] = plane_id
    v2_tables.append(table)

v2 = pd.concat(v2_tables).reset_index().merge(aircraft, on="icao24", how="left")
v2["flag"] = v2["back_to_back"] >= 1
v2["start_utc"] = pd.to_datetime(v2["start_utc"], unit="s")

print("Approaches:", len(v2))
print("Flags, new rule:", v2["flag"].sum())
v2[v2["flag"]][["icao24", "model", "start_utc", "max_sink_fpm", "back_to_back"]]

In [ ]:
import matplotlib.pyplot as plt

# ac26ef track, frozen rows removed
p = day[day["icao24"] == "ac26ef"].sort_values("time")
frozen = (
    (p["alt_ft"] == p["alt_ft"].shift()) &
    (p["speed_kt"] == p["speed_kt"].shift()) &
    (p["vs_fpm"] == p["vs_fpm"].shift())
)
p = p[~frozen].copy()

# Cut into approaches, same rule as before
p["descending"] = (p["vs_fpm"] < -100) & (p["alt_ft"] < 700)
starts = p["descending"] & ~p["descending"].shift(fill_value=False)
p["approach_id"] = starts.cumsum()
d = p[p["descending"]].copy()
d["utc"] = pd.to_datetime(d["time"], unit="s")

# One line per approach
stats = d.groupby("approach_id").agg(
    start=("utc", "min"),
    points=("time", "count"),
    worst=("vs_fpm", "min"),
)

# Flagged approach starts at 15:05:00. Clean approach: the longest one, never over 900 fpm
flag_id = stats[stats["start"] == "2022-06-27 15:05:00"].index[0]
clean_id = stats[stats["worst"] > -900].sort_values("points").index[-1]
print(stats.loc[[flag_id, clean_id]])

# Three charts stacked: altitude, speed, sink rate
fig, ax = plt.subplots(3, 1, figsize=(9, 8), sharex=True)
for aid, label, color, style in [
    (flag_id, "Flagged approach", "#eb6834", "-"),
    (clean_id, "Clean approach", "#2a78d6", "--"),
]:
    a = d[d["approach_id"] == aid]
    sec = a["time"] - a["time"].min()
    name = label + ", " + a["utc"].min().strftime("%H:%M UTC")
    ax[0].plot(sec, a["alt_ft"], style, color=color, lw=2, marker="o", ms=5, label=name)
    ax[1].plot(sec, a["speed_kt"], style, color=color, lw=2, marker="o", ms=5)
    ax[2].plot(sec, -a["vs_fpm"], style, color=color, lw=2, marker="o", ms=5)

ax[2].axhline(1000, color="gray", lw=1, ls=":")
ax[2].text(0, 1020, "1,000 fpm limit", color="gray", fontsize=9)

ax[0].set_ylabel("Altitude (ft)")
ax[1].set_ylabel("Ground speed (kt)")
ax[2].set_ylabel("Sink rate (fpm)")
ax[2].set_xlabel("Seconds since approach start")
ax[0].legend(frameon=False)
for one in ax:
    one.grid(alpha=0.3)
    one.spines[["top", "right"]].set_visible(False)

fig.suptitle("Same Cessna 172S, same day: flagged vs clean approach (KDAB, 27 June 2022)")
plt.tight_layout()
plt.savefig("approach_compare.png", dpi=150)
plt.show()

In [ ]:
def clean_and_cut(df):
    """Remove frozen rows, cut the track into approaches, return descending rows only."""
    df = df.sort_values("time")
    frozen = (
        (df["alt_ft"] == df["alt_ft"].shift()) &
        (df["speed_kt"] == df["speed_kt"].shift()) &
        (df["vs_fpm"] == df["vs_fpm"].shift())
    )
    df = df[~frozen].copy()
    df["descending"] = (df["vs_fpm"] < -100) & (df["alt_ft"] < 700)
    starts = df["descending"] & ~df["descending"].shift(fill_value=False)
    df["approach_id"] = starts.cumsum()
    return df[df["descending"]].copy()


def profile(values):
    """Turn numbers into short text, for example: 1280, 1088, 1152"""
    return ", ".join(str(int(round(v))) for v in values)


rows = []
for plane_id in pattern_ids:
    d = clean_and_cut(day[day["icao24"] == plane_id])
    for aid, a in d.groupby("approach_id"):
        if len(a) < 3:
            continue
        high = (a["vs_fpm"] < -1000) & (a["alt_ft"] > 100)
        b2b = (high & high.shift(fill_value=False) & (a["time"].diff() <= 15)).sum()
        rows.append({
            "icao24": plane_id,
            "start_utc": pd.to_datetime(a["time"].min(), unit="s"),
            "duration_s": int(a["time"].max() - a["time"].min()),
            "altitude_ft": profile(a["alt_ft"]),
            "ground_speed_kt": profile(a["speed_kt"]),
            "sink_rate_fpm": profile(-a["vs_fpm"]),
            "rule_b_flag": b2b >= 1,
        })

all_appr = pd.DataFrame(rows).merge(aircraft[["icao24", "model"]], on="icao24", how="left")

# Keep training aircraft only
trainers = all_appr["model"].fillna("").str.contains("172|PA-28|DA 40|DA 42|182|152")
all_appr = all_appr[trainers]

# Shuffle the order and give each approach a neutral code
all_appr = all_appr.sample(frac=1, random_state=42).reset_index(drop=True)
all_appr["approach_code"] = ["A" + str(i + 1).zfill(3) for i in range(len(all_appr))]

# Sheet for instructors: no aircraft ID, no time, no rule result
sheet = all_appr[["approach_code", "model", "duration_s",
                  "altitude_ft", "ground_speed_kt", "sink_rate_fpm"]].copy()
sheet["label"] = ""       # stable, unstable, or unsure
sheet["reason"] = ""
sheet["reviewer"] = ""
sheet.to_csv("labeling_sheet.csv", index=False)

# Private key: links each code back to aircraft, time, and rule result
all_appr.to_csv("labeling_key.csv", index=False)

print("Approaches on the sheet:", len(sheet))
print("Rule B flags, hidden in the key:", all_appr["rule_b_flag"].sum())
sheet.head(5)